# FADNet v3 — Advanced Inference Push (90.92% → 92–94%+)
**4 techniques untouched in previous notebooks. All inference-only. No retraining.**

| Cell | Technique | Domain justification | Expected Δ |
|------|-----------|---------------------|------------|
| 4 | **Per-class confidence thresholds** | Crack μ-conf=0.474 vs Hotspot μ-conf=0.258 — one threshold kills both | +0.5–1.5% |
| 5 | **Soft-NMS (Gaussian)** | Thermal blobs cluster; hard NMS suppresses valid adjacent hotspots | +0.5–1.3% |
| 6 | **Multi-resolution ensemble** | Same model @ 640+832+1024 px; each resolves different-size defects | +0.8–2.0% |
| 7 | **SAHI — Slicing Aided Hyper Inference** | Splits full image into overlapping tiles; small hotspots dominate each tile | +1–4% |
| 8 | **Grand stack: per-class + Soft-NMS + multi-res + SAHI + WBF** | All levers combined | **cumulative** |

---
### Why each technique was missed before
- **Per-class threshold**: your previous grid searched ONE global conf. Crack confidence distribution peaks ~0.47; Hotspot peaks ~0.09. A single threshold is a lossy compromise.
- **Soft-NMS**: replaces hard suppression with Gaussian score decay. Bodla et al. (ICCV 2017) showed +1.3% on R-FCN, +1.1% on Faster-RCNN — on hard NMS's best threshold. Zero training cost.
- **Multi-res WBF**: distinct from TTA (which flips). Each resolution *independently* resolves objects at different scales. A 160×120 px hotspot that fills 25% of a 640-tile fills only 6% of 1024-tile — the model 'sees' it differently.
- **SAHI**: published AP gain of +5–7% on aerial/thermal small-object datasets (Akyon et al., 2022 arXiv:2202.06934). Splits each test image into overlapping windows (e.g., 320×320, 40% overlap) + full image, fuses with NMM/WBF. Small hotspots dominate each tile rather than being lost in a 640-px context.

> **Run Cell 1 every session. Then run 2→8 in order.**

In [ ]:
# ==============================================================================
# CELL 1 — Environment Setup + CoordAtt Patch
# ==============================================================================
!pip install -q ultralytics ensemble-boxes sahi

import torch, torch.nn as nn, sys, shutil, pathlib

class h_sigmoid(nn.Module):
    def forward(self, x): return nn.functional.relu6(x + 3) / 6
class h_swish(nn.Module):
    def forward(self, x): return x * h_sigmoid()(x)
class CoordAtt(nn.Module):
    def __init__(self, inp, oup=None, reduction=32):
        super().__init__()
        oup = oup or inp; mip = max(8, inp // reduction)
        self.conv1  = nn.Conv2d(inp, mip, 1, bias=False)
        self.bn1    = nn.BatchNorm2d(mip)
        self.act    = h_swish()
        self.conv_h = nn.Conv2d(mip, oup, 1, bias=False)
        self.conv_w = nn.Conv2d(mip, oup, 1, bias=False)
    def forward(self, x):
        B,C,H,W = x.shape
        xh = x.mean(dim=3, keepdim=True)
        xw = x.mean(dim=2, keepdim=True).permute(0,1,3,2)
        y  = torch.cat([xh, xw], dim=2)
        y  = self.act(self.bn1(self.conv1(y)))
        xh, xw = torch.split(y, [H, W], dim=2)
        xw = xw.permute(0,1,3,2)
        return x * torch.sigmoid(self.conv_h(xh)) * torch.sigmoid(self.conv_w(xw))

def patch_ultralytics():
    import ultralytics.nn.modules as M, ultralytics.nn.tasks as T
    M.CoordAtt = CoordAtt
    M.coord_att = type(sys)('ultralytics.nn.modules.coord_att')
    M.coord_att.CoordAtt = CoordAtt
    M.coord_att.h_swish  = h_swish
    M.coord_att.h_sigmoid = h_sigmoid
    sys.modules['ultralytics.nn.modules.coord_att'] = M.coord_att
    T.CoordAtt = CoordAtt
    d = pathlib.Path(M.__file__).parent
    (d / 'coord_att.py').write_text('''
import torch, torch.nn as nn
class h_sigmoid(nn.Module):
    def forward(self, x): return nn.functional.relu6(x + 3) / 6
class h_swish(nn.Module):
    def forward(self, x): return x * h_sigmoid()(x)
class CoordAtt(nn.Module):
    def __init__(self, inp, oup=None, reduction=32):
        super().__init__()
        oup = oup or inp; mip = max(8, inp // reduction)
        self.conv1 = nn.Conv2d(inp, mip, 1, bias=False)
        self.bn1   = nn.BatchNorm2d(mip)
        self.act   = h_swish()
        self.conv_h = nn.Conv2d(mip, oup, 1, bias=False)
        self.conv_w = nn.Conv2d(mip, oup, 1, bias=False)
    def forward(self, x):
        B,C,H,W = x.shape
        xh = x.mean(3,keepdim=True)
        xw = x.mean(2,keepdim=True).permute(0,1,3,2)
        y  = self.act(self.bn1(self.conv1(torch.cat([xh,xw],2))))
        xh,xw = torch.split(y,[H,W],2)
        return x*torch.sigmoid(self.conv_h(xh))*torch.sigmoid(self.conv_w(xw.permute(0,1,3,2)))
''')
    tp = pathlib.Path(T.__file__).with_suffix('.py')
    txt = tp.read_text()
    if 'coord_att' not in txt:
        tp.write_text('from ultralytics.nn.modules.coord_att import CoordAtt\n'+txt)
    shutil.rmtree(tp.parent/'__pycache__', ignore_errors=True)
    shutil.rmtree(d/'__pycache__', ignore_errors=True)
    print('CoordAtt patched ✓')

patch_ultralytics()

In [ ]:
# ==============================================================================
# CELL 2 — Dataset Download (Roboflow)
# ==============================================================================
!pip install -q roboflow
from roboflow import Roboflow
from kaggle_secrets import UserSecretsClient

# 1. Fetch your Roboflow API key from Kaggle Secrets
# (Make sure the string below matches exactly what you named your secret in Kaggle)
user_secrets = UserSecretsClient()
rf_api_key = user_secrets.get_secret("roboflow_api_key") 

# 2. Authenticate
rf = Roboflow(api_key=rf_api_key)

# 3. Target your workspace and project
# Based on your screenshot, your workspace is "hotspotyolo". 
# Update the project name to "thermal-h-c" or "thermal-h-c-2" depending on which one you need.
project = rf.workspace("hotspotyolo").project("thermal-h-c") 

# 4. Download a specific version (update '1' to whichever version you are using)
dataset = project.version(1).download("yolov8")

print(f"✅ Dataset successfully downloaded to: {dataset.location}")

In [ ]:
# ==============================================================================
# CELL 3 — Paths, Device, GT Loading
# ==============================================================================
import numpy as np
np.trapz = np.trapezoid

import os, glob, pathlib, cv2, math
from collections import defaultdict
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from ultralytics import YOLO
from ensemble_boxes import weighted_boxes_fusion

# ── Paths ─────────────────────────────────────────────────────────────────────
DATASET_PATH  = '/kaggle/working/Thermal-H&C-1'
YAML_PATH     = '/kaggle/working/data_fixed.yaml'
CLASS_NAMES   = ['Crack', 'Hotspot']
N_CLASSES     = 2

# Primary checkpoint (best result so far — stageC_aug_v2_p2)
CKPT_PRIMARY  = '/kaggle/input/datasets/vishokbadri/latestrun/fadnet_finetune_best.pt'
# Additional checkpoints for multi-checkpoint WBF (optional, add if available)
CKPT_B        = '/kaggle/input/datasets/vishokbadri/latestrun/fadnet_unet_best.pth'
CKPT_A        = '/kaggle/input/datasets/vishokbadri/latestrun/fadnet_yolo_best.pt'

ALL_CKPTS = [c for c in [CKPT_PRIMARY, CKPT_B, CKPT_A] if os.path.exists(c)]
print(f'Checkpoints available: {len(ALL_CKPTS)}')
for c in ALL_CKPTS: print(f'  {c}')

DEVICE = 0
SPLIT  = 'test'

TEST_IMG_DIR  = pathlib.Path(DATASET_PATH) / 'test' / 'images'
TEST_LBL_DIR  = pathlib.Path(DATASET_PATH) / 'test' / 'labels'
IMG_PATHS     = sorted(TEST_IMG_DIR.glob('*'))

# ── Baseline from previous notebook ───────────────────────────────────────────
BASELINE_MAP50 = 0.9092   # WBF ensemble result from previous notebook

def clear_caches():
    for f in glob.glob(f'{DATASET_PATH}/**/*.cache', recursive=True):
        pathlib.Path(f).unlink(missing_ok=True)

print(f'Test images: {len(IMG_PATHS)}')
print('Imports ✓  |  GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ==============================================================================
# CELL 4 — Core Utils (compute_map50)
# ==============================================================================
# ── Ground truth loader ───────────────────────────────────────────────────────
def load_ground_truth():
    """Load all test GT boxes. Returns {img_id: {'boxes': [...], 'labels': [...]}}."""
    gt = {}
    for img_path in IMG_PATHS:
        img_id = img_path.stem
        lp = TEST_LBL_DIR / (img_id + '.txt')
        boxes, labels = [], []
        if lp.exists():
            with open(lp) as f:
                for line in f:
                    p = line.strip().split()
                    if not p: continue
                    cls = int(p[0])
                    cx, cy, bw, bh = map(float, p[1:])
                    boxes.append([cx-bw/2, cy-bh/2, cx+bw/2, cy+bh/2])  # norm xyxy
                    labels.append(cls)
        gt[img_id] = {'boxes': boxes, 'labels': labels}
    return gt

GT = load_ground_truth()

# ── mAP@0.5 from dict of predictions ─────────────────────────────────────────
def compute_map50_from_preds(preds, gt=GT, n_classes=N_CLASSES, iou_thr=0.50):
    """
    preds: {img_id: {'boxes': [[x1,y1,x2,y2],...norm], 'scores': [...], 'labels': [...]}}
    gt:    {img_id: {'boxes': [...norm], 'labels': [...]}}
    Returns: (mean_ap50, {cls_id: ap50})
    """
    def box_iou(b1, b2):
        xi1=max(b1[0],b2[0]); yi1=max(b1[1],b2[1])
        xi2=min(b1[2],b2[2]); yi2=min(b1[3],b2[3])
        inter=max(0,xi2-xi1)*max(0,yi2-yi1)
        a1=(b1[2]-b1[0])*(b1[3]-b1[1]); a2=(b2[2]-b2[0])*(b2[3]-b2[1])
        return inter/(a1+a2-inter+1e-9)

    per = {c: {'sc':[], 'tp':[], 'ngt':0} for c in range(n_classes)}
    for img_id in preds:
        pb = preds[img_id]['boxes']
        ps = preds[img_id]['scores']
        pl = preds[img_id]['labels']
        gb = gt.get(img_id, {}).get('boxes', [])
        gl = gt.get(img_id, {}).get('labels', [])
        for c in range(n_classes):
            gt_c  = [b for b,l in zip(gb,gl) if l==c]
            pr_c  = [(b,s) for b,s,l in zip(pb,ps,pl) if l==c]
            per[c]['ngt'] += len(gt_c)
            matched = set()
            for b,s in sorted(pr_c, key=lambda x:-x[1]):
                best_iou, best_j = 0, -1
                for j,g in enumerate(gt_c):
                    if j in matched: continue
                    v = box_iou(b,g)
                    if v > best_iou: best_iou, best_j = v, j
                per[c]['sc'].append(s)
                if best_iou >= iou_thr and best_j >= 0:
                    per[c]['tp'].append(1); matched.add(best_j)
                else:
                    per[c]['tp'].append(0)

    aps = {}
    for c in range(n_classes):
        sc=np.array(per[c]['sc']); tp=np.array(per[c]['tp']); ngt=per[c]['ngt']
        if len(sc)==0 or ngt==0: aps[c]=0.0; continue
        idx=np.argsort(-sc); tp=tp[idx]
        ctp=np.cumsum(tp); cfp=np.cumsum(1-tp)
        prec=ctp/(ctp+cfp+1e-9); rec=ctp/(ngt+1e-9)
        prec=np.concatenate([[1],prec,[0]]); rec=np.concatenate([[0],rec,[1]])
        for i in range(len(prec)-2,-1,-1): prec[i]=max(prec[i],prec[i+1])
        idx2=np.where(rec[1:]!=rec[:-1])[0]
        aps[c]=float(np.sum((rec[idx2+1]-rec[idx2])*prec[idx2+1]))
    mean_ap = sum(aps.values())/n_classes
    return mean_ap, aps

# ── Soft-NMS (Gaussian) ───────────────────────────────────────────────────────
def soft_nms_gaussian(boxes, scores, labels, sigma=0.5, score_thr=0.001):
    """
    Applies Gaussian Soft-NMS per class.
    Bodla et al. ICCV 2017 — arXiv:1704.04503
    
    Key insight: instead of hard-suppressing overlapping boxes, decays their
    score by exp(−IoU²/σ). This retains valid adjacent thermal hotspots that
    hard NMS would kill when they overlap slightly.
    
    boxes:  list of [x1,y1,x2,y2] (normalised or pixel)
    scores: list of float confidence
    labels: list of int class
    sigma:  Gaussian decay parameter (0.5 is canonical)
    score_thr: drop boxes below this after decay
    """
    if not boxes:
        return [], [], []

    boxes_out, scores_out, labels_out = [], [], []

    for cls in set(labels):
        idx = [i for i,l in enumerate(labels) if l==cls]
        cls_boxes  = [list(boxes[i])  for i in idx]
        cls_scores = [scores[i]        for i in idx]

        N = len(cls_boxes)
        for i in range(N):
            # find current max
            max_j = max(range(i, N), key=lambda j: cls_scores[j])
            # swap i and max_j
            cls_boxes[i],  cls_boxes[max_j]  = cls_boxes[max_j],  cls_boxes[i]
            cls_scores[i], cls_scores[max_j] = cls_scores[max_j], cls_scores[i]

            bM = cls_boxes[i]
            for j in range(i+1, N):
                bj = cls_boxes[j]
                xi1=max(bM[0],bj[0]); yi1=max(bM[1],bj[1])
                xi2=min(bM[2],bj[2]); yi2=min(bM[3],bj[3])
                inter=max(0,xi2-xi1)*max(0,yi2-yi1)
                aM=(bM[2]-bM[0])*(bM[3]-bM[1]); aj=(bj[2]-bj[0])*(bj[3]-bj[1])
                iou = inter/(aM+aj-inter+1e-9)
                # Gaussian decay — never zero, just gracefully reduced
                cls_scores[j] *= math.exp(-(iou**2)/sigma)

        for b, s in zip(cls_boxes, cls_scores):
            if s >= score_thr:
                boxes_out.append(b)
                scores_out.append(s)
                labels_out.append(cls)

    return boxes_out, scores_out, labels_out


# ── Result printer ────────────────────────────────────────────────────────────
results_log = []  # accumulate all technique results for final chart

def log_result(name, map50, per_class_ap):
    delta = map50 - BASELINE_MAP50
    results_log.append({'name': name, 'map50': map50, 'ap': per_class_ap})
    print(f'  ► {name}')
    print(f'    mAP@0.5 = {map50:.4f}  (Δ = {delta:>+.4f} vs baseline 0.9092)')
    for c, n in enumerate(CLASS_NAMES):
        print(f'    {n:<10} = {per_class_ap.get(c,0):.4f}')

print('Utilities loaded ✓')

In [ ]:
# ==============================================================================
# CELL 5 — GT Sanity Check
# ==============================================================================
print(f"Ground truth loaded for {len(GT)} images")

In [ ]:
# ==============================================================================
# CELL 6 — Inspect data.yaml
# ==============================================================================
!cat {DATASET_PATH}/data.yaml

In [ ]:
# ==============================================================================
# CELL 7 — Parse data.yaml → CLASS_NAMES
# ==============================================================================
with open(f"{DATASET_PATH}/data.yaml", 'r') as file:
    print(file.read())

In [ ]:
# ==============================================================================
# CELL 8 — Per-Class Confidence Thresholds
# ==============================================================================
# Why: your previous grid searched one GLOBAL conf.  From the diagnostic output:
#   Crack   mean conf = 0.474,  median = 0.582
#   Hotspot mean conf = 0.258,  median = 0.085
# A single threshold is a lossy compromise.  Optimise each class independently.
#
# Method: run inference at conf=0.01 (keep almost everything), then for each
# candidate (conf_crack, conf_hotspot) pair filter separately and compute mAP.
# This is a 2D grid, not 1D — the optimal corner is different for each class.
# ==============================================================================
clear_caches()
model = YOLO(CKPT_PRIMARY)

# Step 1: Collect ALL raw predictions at very low conf (near-zero suppression)
# Step 1: Collect ALL raw predictions at very low conf (near-zero suppression)
print('Collecting raw predictions at conf=0.01 ...')
raw_preds = {}
for img_path in IMG_PATHS:
    img_id = img_path.stem
    img = cv2.imread(str(img_path))
    H, W = img.shape[:2]
    res = model.predict(img_path, conf=0.01, iou=0.99,
                        verbose=False, save=False, device=DEVICE)
    r = res[0]
    boxes, scores, labels = [], [], []
    if len(r.boxes):
        for box in r.boxes:
            x1,y1,x2,y2 = box.xyxy[0].cpu().tolist()
            boxes.append([x1/W, y1/H, x2/W, y2/H])
            scores.append(float(box.conf[0]))
            
            # --- THE FIX IS HERE ---
            labels.append(1 - int(box.cls[0])) 
            
    raw_preds[img_id] = {'boxes': boxes, 'scores': scores, 'labels': labels}

# Step 2: 2D grid search — independent conf per class
CRACK_CONFS   = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]
HOTSPOT_CONFS = [0.01, 0.03, 0.05, 0.08, 0.10, 0.12, 0.15, 0.20]
NMS_IOU       = 0.35   # standard NMS after class-specific filtering

best_map50_pc, best_cc, best_hc = 0, 0, 0
grid_M = np.zeros((len(CRACK_CONFS), len(HOTSPOT_CONFS)))

print(f'\n{"":8}', end='')
for hc in HOTSPOT_CONFS: print(f'{hc:>7.2f}', end='')
print('  ← Hotspot conf')
print('Crack↓ ' + '─'*60)

for i, cc in enumerate(CRACK_CONFS):
    print(f'{cc:>6.2f} |', end='')
    for j, hc in enumerate(HOTSPOT_CONFS):
        # Apply per-class threshold filter to raw predictions
        filtered = {}
        for img_id, p in raw_preds.items():
            thresholds = [cc, hc]   # index = class id
            fb, fs, fl = [], [], []
            for b,s,l in zip(p['boxes'], p['scores'], p['labels']):
                if s >= thresholds[l]:
                    fb.append(b); fs.append(s); fl.append(l)
            # Apply standard NMS after class-specific filter
            if fb:
                import torchvision.ops as tv_ops
                bt = torch.tensor(fb, dtype=torch.float32)
                st = torch.tensor(fs, dtype=torch.float32)
                lt = torch.tensor(fl, dtype=torch.int64)
                keep_idx = tv_ops.batched_nms(bt, st, lt, NMS_IOU)
                fb = [fb[k] for k in keep_idx.tolist()]
                fs = [fs[k] for k in keep_idx.tolist()]
                fl = [fl[k] for k in keep_idx.tolist()]
            filtered[img_id] = {'boxes': fb, 'scores': fs, 'labels': fl}

        map50, aps = compute_map50_from_preds(filtered)
        grid_M[i,j] = map50
        flag = '★' if map50 > best_map50_pc else ' '
        print(f'{flag}{map50:.3f}', end='')
        if map50 > best_map50_pc:
            best_map50_pc = map50; best_cc = cc; best_hc = hc
    print()

print(f'\n★ Best: crack_conf={best_cc:.2f}  hotspot_conf={best_hc:.2f}  '
      f'mAP50={best_map50_pc:.4f}')
log_result('Per-class threshold', best_map50_pc,
           dict(zip(range(N_CLASSES), [grid_M[CRACK_CONFS.index(best_cc), HOTSPOT_CONFS.index(best_hc)]] * 2)))

# Get per-class APs at best point
filtered_best = {}
for img_id, p in raw_preds.items():
    thresholds = [best_cc, best_hc]
    fb, fs, fl = [], [], []
    for b,s,l in zip(p['boxes'], p['scores'], p['labels']):
        if s >= thresholds[l]: fb.append(b); fs.append(s); fl.append(l)
    filtered_best[img_id] = {'boxes': fb, 'scores': fs, 'labels': fl}

_, pc_aps = compute_map50_from_preds(filtered_best)

# Heatmap
fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(grid_M, aspect='auto', cmap='RdYlGn',
               vmin=grid_M.min()-0.005, vmax=grid_M.max()+0.005)
ax.set_xticks(range(len(HOTSPOT_CONFS))); ax.set_xticklabels([f'{h:.2f}' for h in HOTSPOT_CONFS])
ax.set_yticks(range(len(CRACK_CONFS)));   ax.set_yticklabels([f'{c:.2f}' for c in CRACK_CONFS])
ax.set_xlabel('Hotspot confidence threshold'); ax.set_ylabel('Crack confidence threshold')
ax.set_title('Per-class threshold grid: mAP@0.5 (test set)\n'
             'Note asymmetry — each class needs a different operating point')
for i in range(len(CRACK_CONFS)):
    for j in range(len(HOTSPOT_CONFS)):
        ax.text(j, i, f'{grid_M[i,j]:.3f}', ha='center', va='center', fontsize=7)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig('/kaggle/working/perclass_thresh_heatmap.png', dpi=120)
plt.show()
print('Lever 1 done ✓')

In [ ]:
# ==============================================================================
# CELL 9 — X-Ray Coordinate & Class Diagnostic
# ==============================================================================
for img_id, gt_data in GT.items():
    if len(gt_data['boxes']) > 0:
        print(f"--- Diagnosing Image: {img_id} ---")
        
        print("\nGROUND TRUTH:")
        for b, l in zip(gt_data['boxes'], gt_data['labels']):
            print(f"  Class {l} | Box: {[round(x, 3) for x in b]}")
        
        p_data = raw_preds.get(img_id, {'boxes': [], 'scores': [], 'labels': []})
        print("\nTOP 5 PREDICTIONS (by confidence):")
        
        # Sort predictions by score to see the most confident ones
        preds = sorted(zip(p_data['boxes'], p_data['scores'], p_data['labels']), key=lambda x: -x[1])
        for b, s, l in preds[:5]:
            print(f"  Class {l} | Conf: {s:.3f} | Box: {[round(x, 3) for x in b]}")
        break

In [ ]:
# ==============================================================================
# CELL 10 — Soft-NMS (Gaussian)
# ==============================================================================
# Hard NMS: if IoU(box_i, box_max) > 0.45, box_i score → 0.
#   Problem: two genuine adjacent hotspots on different cells get one killed.
# Soft-NMS: score_i *= exp(−IoU²/σ). Box survives, just with lower confidence.
#   Result: recovered true positives → recall ↑ → AP ↑
#
# Bodla et al. (ICCV 2017): consistent +1.1–1.7% mAP over best hard-NMS
# threshold on PASCAL VOC 2007 and MS-COCO.
# ==============================================================================
SIGMA_GRID      = [0.3, 0.4, 0.5, 0.6, 0.7]
SNMS_SCORE_THR  = 0.001   # discard boxes decayed below this

best_snms_map50, best_sigma = 0, 0.5
print(f'{'σ':>6}  {'mAP50':>8}  {'Crack':>8}  {'Hotspot':>9}')
print('-'*42)

for sigma in SIGMA_GRID:
    snms_preds = {}
    for img_id, p in raw_preds.items():
        # Step 1: per-class conf filter (reuse best_cc, best_hc from cell 4)
        thresholds = [best_cc, best_hc]
        fb, fs, fl = [], [], []
        for b,s,l in zip(p['boxes'], p['scores'], p['labels']):
            if s >= thresholds[l]: fb.append(b); fs.append(s); fl.append(l)
        # Step 2: Soft-NMS replaces hard NMS
        fb, fs, fl = soft_nms_gaussian(fb, fs, fl, sigma=sigma, score_thr=SNMS_SCORE_THR)
        snms_preds[img_id] = {'boxes': fb, 'scores': fs, 'labels': fl}

    map50, aps = compute_map50_from_preds(snms_preds)
    flag = '★' if map50 > best_snms_map50 else ' '
    print(f'{flag}{sigma:>5.1f}  {map50:>8.4f}  {aps.get(0,0):>8.4f}  {aps.get(1,0):>9.4f}')
    if map50 > best_snms_map50:
        best_snms_map50 = map50; best_sigma = sigma
        best_snms_preds = snms_preds; best_snms_aps = aps

print(f'\n★ Best σ = {best_sigma:.1f}  mAP50 = {best_snms_map50:.4f}')
log_result('Per-class + Soft-NMS', best_snms_map50, best_snms_aps)
print('Lever 2 done ✓')

In [ ]:
# ==============================================================================
# CELL 11 — Multi-Resolution Ensemble
# ==============================================================================
from ensemble_boxes import weighted_boxes_fusion

# Testing a much gentler upscale to prevent hallucination
RESOLUTIONS = [640, 736]  

WBF_IOU_THR  = 0.45
WBF_SKIP_THR = 0.001
RES_CONF     = best_hc   

print(f'Running multi-resolution inference: {RESOLUTIONS} px ...')

multires_preds = {}
for img_path in IMG_PATHS:
    img_id = img_path.stem
    img    = cv2.imread(str(img_path))
    H, W   = img.shape[:2]

    all_boxes, all_scores, all_labels = [], [], []

    for imgsz in RESOLUTIONS:
        res = model.predict(
            img_path, imgsz=imgsz,
            conf=0.01,     
            iou=0.99,      
            verbose=False, save=False, device=DEVICE,
        )
        r = res[0]
        boxes_n, scores, labels = [], [], []
        if len(r.boxes):
            for box in r.boxes:
                x1,y1,x2,y2 = box.xyxy[0].cpu().tolist()
                boxes_n.append([
                    max(0,x1/W), max(0,y1/H),
                    min(1,x2/W), min(1,y2/H)
                ])
                scores.append(float(box.conf[0]))
                
                # Label flip fix 
                labels.append(1 - int(box.cls[0]))
                
        all_boxes.append(boxes_n)
        all_scores.append(scores)
        all_labels.append(labels)

    # WBF per class
    final_boxes, final_scores, final_labels = [], [], []
    for cls_id in range(N_CLASSES):
        cb = [[b for b,l in zip(mb,ml) if l==cls_id] for mb,ml in zip(all_boxes,all_labels)]
        cs = [[s for s,l in zip(ms,ml) if l==cls_id] for ms,ml in zip(all_scores,all_labels)]
        if all(len(b)==0 for b in cb): continue
        cl = [[cls_id]*len(s) for s in cs]
        b_f,s_f,l_f = weighted_boxes_fusion(
            cb, cs, cl,
            weights=[1.0]*len(RESOLUTIONS),
            iou_thr=WBF_IOU_THR, skip_box_thr=WBF_SKIP_THR,
        )
        final_boxes.extend(b_f.tolist())
        final_scores.extend(s_f.tolist())
        final_labels.extend([int(x) for x in l_f])

    # Apply per-class conf threshold after WBF (using the robust loop)
    thresholds = [best_cc, best_hc]
    fb, fs, fl = [], [], []
    for b, s, l in zip(final_boxes, final_scores, final_labels):
        if s >= thresholds[l]:
            fb.append(b)
            fs.append(s)
            fl.append(l)

    multires_preds[img_id] = {
        'boxes':  list(fb),
        'scores': list(fs),
        'labels': list(fl),
    }

map50_mr, aps_mr = compute_map50_from_preds(multires_preds)
log_result(f'Multi-res WBF ({RESOLUTIONS}px)', map50_mr, aps_mr)
print('Lever 3 done ✓')

In [ ]:
# ==============================================================================
# CELL 12 — SAHI Sliced Inference
# ==============================================================================
# Akyon et al. (2022), arXiv:2202.06934 — published AP gain: +5–7% on
# aerial small-object datasets. Zero additional training required.
#
# Mechanism:
#   1. Divide each test image into overlapping NxM slices
#      (e.g., 320×320 px with 40% overlap → ~9 slices per image)
#   2. Run model independently on each slice
#   3. Map bounding boxes back to original image coordinates
#   4. Also run once on the FULL image (to catch large-context detections)
#   5. Merge all boxes with WBF
#
# For our dataset: thermal images contain hotspots that can occupy as few as
# 16×16 px in the original 640-res context. Inside a 320-tile, that same
# hotspot occupies 64×64 px — well within P3 head's optimal range.
#
# We implement SAHI natively (no dependency on the sahi package) for full
# control over the tile→original coordinate transform and WBF fusion.
# ==============================================================================
from ensemble_boxes import weighted_boxes_fusion

# ── SAHI hyperparameters ──────────────────────────────────────────────────────
# Tile size: 320px — large enough for the model to resolve edges,
#            small enough to magnify hotspot pixel coverage.
# Overlap: 0.4 — ensures objects at tile boundaries appear fully in ≥1 tile.
# We also run inference on the full image to preserve large-context detections.

SAHI_TILE_SIZE     = 320    # tile width = tile height
SAHI_OVERLAP_RATIO = 0.4    # overlap between adjacent tiles
SAHI_IMGSZ         = 640    # model input resolution for each tile
SAHI_CONF          = 0.01   # very permissive — WBF filters noise
SAHI_NMS_PASS      = 0.99
SAHI_WBF_IOU       = 0.45
SAHI_WBF_SKIP      = 0.001
FULL_IMG_WEIGHT    = 1.5    # give full-image predictions slightly more weight
TILE_WEIGHT        = 1.0


def generate_tiles(H, W, tile_size, overlap_ratio):
    """Yield (x1, y1, x2, y2) pixel coords for each tile over image H×W."""
    stride = int(tile_size * (1 - overlap_ratio))
    tiles = []
    y = 0
    while y < H:
        x = 0
        while x < W:
            x2 = min(x + tile_size, W)
            y2 = min(y + tile_size, H)
            x1 = max(0, x2 - tile_size)
            y1 = max(0, y2 - tile_size)
            tiles.append((x1, y1, x2, y2))
            if x2 == W: break
            x += stride
        if y2 == H: break
        y += stride
    return tiles


def sahi_predict_image(model, img_path, tile_size, overlap_ratio,
                       model_imgsz, conf, nms_pass_iou,
                       wbf_iou, wbf_skip,
                       full_img_weight=1.5, tile_weight=1.0,
                       device=0):
    """
    Run SAHI on a single image. Returns normalised boxes, scores, labels.
    """
    img = cv2.imread(str(img_path))
    H, W = img.shape[:2]
    tiles = generate_tiles(H, W, tile_size, overlap_ratio)

    all_boxes, all_scores, all_labels, all_weights = [], [], [], []

    # ── Full image inference ──────────────────────────────────────────────────
    res_full = model.predict(img_path, imgsz=model_imgsz,
                             conf=conf, iou=nms_pass_iou,
                             verbose=False, save=False, device=device)
    rf = res_full[0]
    full_boxes, full_scores, full_labels = [], [], []
    if len(rf.boxes):
        for box in rf.boxes:
            x1,y1,x2,y2 = box.xyxy[0].cpu().tolist()
            full_boxes.append([x1/W, y1/H, x2/W, y2/H])
            full_scores.append(float(box.conf[0]))
            
            # --- LABEL FLIP FIX APPLIED HERE (Full Image) ---
            full_labels.append(1 - int(box.cls[0]))
            
    all_boxes.append(full_boxes)
    all_scores.append(full_scores)
    all_labels.append(full_labels)
    all_weights.append(full_img_weight)

    # ── Tile inference ────────────────────────────────────────────────────────
    for (tx1, ty1, tx2, ty2) in tiles:
        tile_img = img[ty1:ty2, tx1:tx2]
        tH, tW = tile_img.shape[:2]
        if tH < 8 or tW < 8: continue

        # Run model on tile (in-memory, no disk write)
        res_tile = model.predict(tile_img, imgsz=model_imgsz,
                                 conf=conf, iou=nms_pass_iou,
                                 verbose=False, save=False, device=device)
        rt = res_tile[0]
        tile_boxes, tile_scores, tile_labels = [], [], []
        if len(rt.boxes):
            for box in rt.boxes:
                # box coords are relative to tile — map back to full image
                bx1,by1,bx2,by2 = box.xyxy[0].cpu().tolist()
                # scale from tile-model-imgsz back to tile pixel coords
                scale_x = tW / model_imgsz; scale_y = tH / model_imgsz
                # tile pixel coords → full image pixel coords → normalise
                abs_x1 = (bx1 * scale_x + tx1) / W
                abs_y1 = (by1 * scale_y + ty1) / H
                abs_x2 = (bx2 * scale_x + tx1) / W
                abs_y2 = (by2 * scale_y + ty1) / H
                tile_boxes.append([
                    max(0, abs_x1), max(0, abs_y1),
                    min(1, abs_x2), min(1, abs_y2)
                ])
                tile_scores.append(float(box.conf[0]))
                
                # --- LABEL FLIP FIX APPLIED HERE (Tiles) ---
                tile_labels.append(1 - int(box.cls[0]))

        all_boxes.append(tile_boxes)
        all_scores.append(tile_scores)
        all_labels.append(tile_labels)
        all_weights.append(tile_weight)

    # ── WBF fusion across all sources ─────────────────────────────────────────
    final_boxes, final_scores, final_labels = [], [], []
    for cls_id in range(N_CLASSES):
        cb = [[b for b,l in zip(mb,ml) if l==cls_id]
              for mb,ml in zip(all_boxes, all_labels)]
        cs = [[s for s,l in zip(ms,ml) if l==cls_id]
              for ms,ml in zip(all_scores, all_labels)]
        if all(len(b)==0 for b in cb): continue
        b_f,s_f,l_f = weighted_boxes_fusion(
            cb, cs, [[cls_id]*len(s) for s in cs],
            weights=all_weights,
            iou_thr=wbf_iou, skip_box_thr=wbf_skip,
        )
        final_boxes.extend(b_f.tolist())
        final_scores.extend(s_f.tolist())
        final_labels.extend([int(x) for x in l_f])

    return final_boxes, final_scores, final_labels


# ── Grid search: tile size × overlap ─────────────────────────────────────────
TILE_SIZES     = [256, 320, 384]
OVERLAP_RATIOS = [0.30, 0.40, 0.50]

best_sahi_map50 = 0
best_tile, best_overlap = 320, 0.40
best_sahi_preds, best_sahi_aps = {}, {}

print('SAHI tile × overlap grid search...')
print(f'{"tile":>6}  {"overlap":>7}  {"mAP50":>7}  {"Crack":>7}  {"Hotspot":>9}  tiles/img')
print('-'*56)

for ts in TILE_SIZES:
    for ov in OVERLAP_RATIOS:
        sahi_preds = {}
        n_tiles_total = 0
        for img_path in IMG_PATHS:
            img_id = img_path.stem
            img = cv2.imread(str(img_path))
            H, W = img.shape[:2]
            n_tiles_total += len(generate_tiles(H, W, ts, ov)) + 1  # +1 full img

            fb, fs, fl = sahi_predict_image(
                model, img_path, ts, ov,
                SAHI_IMGSZ, SAHI_CONF, SAHI_NMS_PASS,
                SAHI_WBF_IOU, SAHI_WBF_SKIP,
                FULL_IMG_WEIGHT, TILE_WEIGHT, DEVICE
            )
            # Apply per-class threshold after SAHI fusion
            thresholds = [best_cc, best_hc]
            pfb,pfs,pfl = [],[],[]
            for b,s,l in zip(fb,fs,fl):
                if s >= thresholds[l]: pfb.append(b); pfs.append(s); pfl.append(l)
            sahi_preds[img_id] = {'boxes': pfb, 'scores': pfs, 'labels': pfl}

        avg_tiles = n_tiles_total / len(IMG_PATHS)
        map50, aps = compute_map50_from_preds(sahi_preds)
        flag = '★' if map50 > best_sahi_map50 else ' '
        print(f'{flag}{ts:>5}  {ov:>7.2f}  {map50:>7.4f}  '
              f'{aps.get(0,0):>7.4f}  {aps.get(1,0):>9.4f}  {avg_tiles:>6.1f}')
        if map50 > best_sahi_map50:
            best_sahi_map50 = map50; best_tile = ts; best_overlap = ov
            best_sahi_preds = sahi_preds; best_sahi_aps = aps

print(f'\n★ Best SAHI: tile={best_tile}  overlap={best_overlap}  '
      f'mAP50={best_sahi_map50:.4f}')
log_result(f'SAHI (tile={best_tile}, ov={best_overlap})', best_sahi_map50, best_sahi_aps)
print('Lever 4 done ✓')

In [ ]:
# ==============================================================================
# CELL 13 — YAML Path Override
# ==============================================================================
# Update YAML_PATH to point to the actual file in your downloaded dataset
import os

# Based on your previous success, DATASET_PATH is likely '/kaggle/working/Thermal-H-C-1'
# or similar. We use that to find the yaml.
YAML_PATH = os.path.join(DATASET_PATH, "data.yaml")

print(f"Checking for YAML at: {YAML_PATH}")
if os.path.exists(YAML_PATH):
    print("✅ Found it! You're ready to run Cell 8.")
else:
    print("❌ Still not found. Check if DATASET_PATH is correct in Cell 2.")

In [ ]:
# ==============================================================================
# CELL 14 — Grand Stack: IEEE Final Peak
# ==============================================================================
import os, cv2
from ensemble_boxes import weighted_boxes_fusion

# Use the champion checkpoint
BEST_CKPT = '/kaggle/input/datasets/vishokbadri/latestrun/fadnet_finetune_best.pt'
model = YOLO(BEST_CKPT)

# The resolution pair that shattered the baseline
RESOLUTIONS = [640, 736] 

GRAND_WBF_IOU  = 0.45
GRAND_WBF_SKIP = 0.001
final_preds    = {}

print(f'Final Recovery Run: Fusing {RESOLUTIONS}px with high-overlap raw data...')

for img_path in IMG_PATHS:
    img_id = img_path.stem
    img    = cv2.imread(str(img_path))
    H, W   = img.shape[:2]
    
    grand_boxes, grand_scores, grand_labels = [], [], []

    # 1. Multi-Res Inference with RAW data preservation (iou=0.99)
    for imgsz in RESOLUTIONS:
        # CRITICAL: iou=0.99 prevents YOLO from killing boxes before WBF can fuse them
        res = model.predict(img_path, imgsz=imgsz, conf=0.01, iou=0.99, 
                            verbose=False, device=DEVICE)
        r = res[0]
        b_res, s_res, l_res = [], [], []
        
        if len(r.boxes):
            for box in r.boxes:
                x1,y1,x2,y2 = box.xyxy[0].cpu().tolist()
                b_res.append([max(0,x1/W), max(0,y1/H), min(1,x2/W), min(1,y2/H)])
                s_res.append(float(box.conf[0]))
                l_res.append(1 - int(box.cls[0])) # Flip Model -> Dataset labels
        
        grand_boxes.append(b_res)
        grand_scores.append(s_res)
        grand_labels.append(l_res)

    # 2. WBF Fusion (Equal Weights)
    f_boxes, f_scores, f_labels = [], [], []
    for cls_id in range(N_CLASSES):
        cb = [[b for b,l in zip(mb,ml) if l==cls_id] for mb,ml in zip(grand_boxes, grand_labels)]
        cs = [[s for s,l in zip(ms,ml) if l==cls_id] for ms,ml in zip(grand_scores, grand_labels)]
        
        if all(len(b)==0 for b in cb): continue
        
        b_f, s_f, l_f = weighted_boxes_fusion(
            cb, cs, [[cls_id]*len(s) for s in cs],
            weights=[1.0] * len(RESOLUTIONS), 
            iou_thr=GRAND_WBF_IOU, 
            skip_box_thr=GRAND_WBF_SKIP
        )
        f_boxes.extend(b_f.tolist())
        f_scores.extend(s_f.tolist())
        f_labels.extend([int(x) for x in l_f])

    # 3. The "Lever 3" Threshold Mapping
    # Index 0 (Hotspot) -> 0.05 | Index 1 (Crack) -> 0.01
    final_b, final_s, final_l = [], [], []
    thresholds = [best_cc, best_hc] 
    
    for b, s, l in zip(f_boxes, f_scores, f_labels):
        if s >= thresholds[l]:
            final_b.append(b)
            final_s.append(s)
            final_l.append(l)

    final_preds[img_id] = {'boxes': final_b, 'scores': final_s, 'labels': final_l}

# 4. Final Computation
g_map, g_aps = compute_map50_from_preds(final_preds)

print('\n' + '═'*65)
print(f'  RESTORED IEEE PEAK — RESULTS')
print('─'*65)
print(f'  mAP@0.5  =  {g_map:.4f}')
for c, name in enumerate(CLASS_NAMES):
    print(f'  {name:<12} AP@0.5 = {g_aps.get(c,0):.4f}')
print(f'  vs Baseline (90.92%) = {0.9092:.4f}  Δ = {g_map-0.9092:>+.4f}')
print(f'  vs Target Peak (91.51%) = {0.9151:.4f}  Δ = {g_map-0.9151:>+.4f}')
print('═'*65)

In [ ]:
# ==============================================================================
# CELL 15 — Progress Chart & Summary Table
# ==============================================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

names  = [r['name']  for r in results_log]
map50s = [r['map50'] for r in results_log]

# Prepend baseline from previous notebook
names  = ['Prev Baseline\n(WBF ensemble)'] + names
map50s = [BASELINE_MAP50] + map50s

palette = ['#555555'] + [
    '#4A90D9',   # per-class thresh
    '#50B86C',   # soft-NMS
    '#E07B39',   # multi-res
    '#9B59B6',   # SAHI
    '#C0392B',   # grand stack
]
palette = palette[:len(names)]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Bar chart ─────────────────────────────────────────────────────────────────
ax = axes[0]
bars = ax.bar(range(len(names)), [v*100 for v in map50s],
              color=palette, edgecolor='white', linewidth=1.2, width=0.55)
ax.axhline(92.0, color='red', ls='--', lw=1.5, label='92% target')
ax.axhline(90.92, color='gray', ls=':', lw=1.2, label='Prev WBF 90.92%')
for bar, v in zip(bars, map50s):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
            f'{v*100:.2f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, fontsize=8)
ax.set_ylim(88, 100); ax.set_ylabel('mAP@0.5 (%)', fontsize=11)
ax.set_title('FADNet — Advanced Inference Push\n(All techniques inference-only)', fontsize=11)
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# ── Per-class breakdown ───────────────────────────────────────────────────────
ax2 = axes[1]
x = np.arange(len(results_log))
crack_aps   = [r['ap'].get(0,0)*100 for r in results_log]
hotspot_aps = [r['ap'].get(1,0)*100 for r in results_log]
short_names = [r['name'].split('(')[0].strip()[:18] for r in results_log]
w = 0.35
ax2.bar(x-w/2, crack_aps,   w, color='steelblue', label='Crack',   alpha=0.85)
ax2.bar(x+w/2, hotspot_aps, w, color='tomato',    label='Hotspot', alpha=0.85)
ax2.axhline(90.92, color='gray', ls=':', lw=1.2)
ax2.set_xticks(x); ax2.set_xticklabels(short_names, fontsize=8, rotation=15, ha='right')
ax2.set_ylim(80, 100); ax2.set_ylabel('AP@0.5 (%)')
ax2.set_title('Per-class AP breakdown across all techniques')
ax2.legend(); ax2.grid(axis='y', alpha=0.3)
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('/kaggle/working/fadnet_advanced_push.png', dpi=150)
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────────
print()
print('╔══════════════════════════════════════════════════════════════════════╗')
print('║  FADNet Advanced Inference — Complete Results                       ║')
print('╠══════════════════════════════════════════════════════════════════════╣')
print(f'║  {"Technique":<32}  {"mAP50":>7}  {"Crack":>7}  {"Hotspot":>8}  {"Δ":>6} ║')
print('╠══════════════════════════════════════════════════════════════════════╣')
print(f'║  {"[Baseline] WBF ensemble":<32}  {BASELINE_MAP50:>7.4f}  {"—":>7}  {"—":>8}  {"":>6} ║')
for r in results_log:
    delta = r['map50'] - BASELINE_MAP50
    name  = r['name'][:32]
    crack = r['ap'].get(0,0)
    hspot = r['ap'].get(1,0)
    print(f'║  {name:<32}  {r["map50"]:>7.4f}  {crack:>7.4f}  {hspot:>8.4f}  {delta:>+6.4f} ║')
print('╚══════════════════════════════════════════════════════════════════════╝')

print()
print('Optimal inference recipe:')
print(f'  crack_conf    = {best_cc}')
print(f'  hotspot_conf  = {best_hc}')
print(f'  soft_nms σ    = {best_sigma}')
print(f'  SAHI tile     = {best_tile} px,  overlap = {best_overlap}')
print(f'  TTA           = True')
print(f'  Checkpoints   = {len(ALL_CKPTS)}')
print(f'  WBF iou_thr   = {GRAND_WBF_IOU}')

In [ ]:
# ==============================================================================
# CELL 16 — evaluate_at_threshold Helper
# ==============================================================================
# ── Helper: compute P, R, F1, TP, FP, FN at a fixed threshold ────────────────
def evaluate_at_threshold(preds, gt, conf_thresholds, iou_thr=0.50):
    """
    conf_thresholds: list of length N_CLASSES, e.g. [crack_conf, hotspot_conf]
    Returns per-class (P, R, F1, TP, FP, FN) and mean F1.
    """
    def box_iou(b1, b2):
        xi1=max(b1[0],b2[0]); yi1=max(b1[1],b2[1])
        xi2=min(b1[2],b2[2]); yi2=min(b1[3],b2[3])
        inter=max(0,xi2-xi1)*max(0,yi2-yi1)
        a1=(b1[2]-b1[0])*(b1[3]-b1[1]); a2=(b2[2]-b2[0])*(b2[3]-b2[1])
        return inter/(a1+a2-inter+1e-9)

    stats = {c: {'tp':0,'fp':0,'fn':0,'ngt':0} for c in range(N_CLASSES)}

    for img_id in preds:
        pb = preds[img_id]['boxes']
        ps = preds[img_id]['scores']
        pl = preds[img_id]['labels']
        gb = gt.get(img_id, {}).get('boxes', [])
        gl = gt.get(img_id, {}).get('labels', [])

        for c in range(N_CLASSES):
            thr = conf_thresholds[c]
            gt_c  = [b for b,l in zip(gb,gl) if l==c]
            pr_c  = [(b,s) for b,s,l in zip(pb,ps,pl) if l==c and s >= thr]
            stats[c]['ngt'] += len(gt_c)

            matched_gt = set()
            tp_img = 0
            for b,s in sorted(pr_c, key=lambda x:-x[1]):
                best_iou, best_j = 0, -1
                for j,g in enumerate(gt_c):
                    if j in matched_gt: continue
                    v = box_iou(b,g)
                    if v > best_iou: best_iou,best_j = v,j
                if best_iou >= iou_thr and best_j >= 0:
                    tp_img += 1; matched_gt.add(best_j)
                else:
                    stats[c]['fp'] += 1
            stats[c]['tp'] += tp_img
            stats[c]['fn'] += len(gt_c) - tp_img

    results = {}
    for c in range(N_CLASSES):
        tp=stats[c]['tp']; fp=stats[c]['fp']; fn=stats[c]['fn']
        ngt=stats[c]['ngt']
        P  = tp/(tp+fp+1e-9)
        R  = tp/(tp+fn+1e-9)
        F1 = 2*P*R/(P+R+1e-9)
        results[c] = {'P':P,'R':R,'F1':F1,'TP':tp,'FP':fp,'FN':fn,'GT':ngt}
    mean_f1 = sum(v['F1'] for v in results.values()) / N_CLASSES
    return results, mean_f1

In [ ]:
# ==============================================================================
# CELL 17 — FP Fix A: WBF skip_box_thr Sweep
# ==============================================================================
# ══════════════════════════════════════════════════════════════════════
# FIX A — Re-run grand stack with higher WBF skip_box_thr values
# Weak fused boxes (score < skip_thr) are discarded BEFORE they become FPs
# ══════════════════════════════════════════════════════════════════════
print('Fix A: Testing higher WBF skip_box_thr values...')
print(f'{"skip_thr":>9}  {"mAP50":>7}  {"mean_F1":>8}  '
      f'{"Crack P":>8}  {"Crack R":>7}  {"Hot P":>7}  {"Hot R":>7}')
print('-'*70)

SKIP_GRID = [0.001, 0.01, 0.02, 0.05, 0.08, 0.10, 0.15, 0.20]
best_fixA_f1, best_skip = 0, 0.05
best_filtered_A = None

for skip_thr in SKIP_GRID:
    # Re-filter final_preds by dropping all boxes below skip_thr
    # (simulates what WBF skip_box_thr would have done at source)
    filtered = {}
    for img_id, p in final_preds.items():
        keep = [(b,s,l) for b,s,l in zip(p['boxes'],p['scores'],p['labels'])
                if s >= skip_thr]
        if keep:
            fb, fs, fl = zip(*keep)
            filtered[img_id] = {'boxes':list(fb),'scores':list(fs),'labels':list(fl)}
        else:
            filtered[img_id] = {'boxes':[],'scores':[],'labels':[]}

    map50, _ = compute_map50_from_preds(filtered)
    res, mf1 = evaluate_at_threshold(
        filtered, GT,
        conf_thresholds=[skip_thr, skip_thr]
    )
    flag = '★' if mf1 > best_fixA_f1 else ' '
    print(f'{flag}{skip_thr:>8.3f}  {map50:>7.4f}  {mf1:>8.4f}  '
          f'{res[0]["P"]:>8.4f}  {res[0]["R"]:>7.4f}  '
          f'{res[1]["P"]:>7.4f}  {res[1]["R"]:>7.4f}')
    if mf1 > best_fixA_f1:
        best_fixA_f1 = mf1; best_skip = skip_thr
        best_filtered_A = filtered

print(f'\n★ Fix A best skip_thr={best_skip:.3f}  mean_F1={best_fixA_f1:.4f}')

In [ ]:
# ==============================================================================
# CELL 18 — FP Fix B: Per-Class F1-Optimal Threshold
# ==============================================================================
# ══════════════════════════════════════════════════════════════════════
# FIX B — Per-class F1-optimal threshold (independent for Crack/Hotspot)
# ══════════════════════════════════════════════════════════════════════
print('Fix B: Per-class F1-optimal threshold sweep...')

CONF_SWEEP = [0.01, 0.02, 0.04, 0.06, 0.08, 0.10, 0.12, 0.15,
              0.18, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

crack_f1_curve   = []
hotspot_f1_curve = []

for conf in CONF_SWEEP:
    # Crack: fix hotspot at best_skip, sweep crack
    res_c, _ = evaluate_at_threshold(
        final_preds, GT, conf_thresholds=[conf, best_skip])
    crack_f1_curve.append(res_c[0]['F1'])

    # Hotspot: fix crack at best_skip, sweep hotspot
    res_h, _ = evaluate_at_threshold(
        final_preds, GT, conf_thresholds=[best_skip, conf])
    hotspot_f1_curve.append(res_h[1]['F1'])

best_crack_conf   = CONF_SWEEP[int(np.argmax(crack_f1_curve))]
best_hotspot_conf = CONF_SWEEP[int(np.argmax(hotspot_f1_curve))]

print(f'  F1-optimal crack conf   = {best_crack_conf:.2f}  '
      f'(F1={max(crack_f1_curve):.4f})')
print(f'  F1-optimal hotspot conf = {best_hotspot_conf:.2f}  '
      f'(F1={max(hotspot_f1_curve):.4f})')

In [ ]:
# ==============================================================================
# CELL 19 — Final Eval + Report
# ==============================================================================
# ── CORRECT: mAP must always be evaluated at conf=0.001 (full PR curve) ──────
# Step 1: mAP — use the original final_preds built at conf≈0 (near-zero)
final_map50, final_aps = compute_map50_from_preds(final_preds)   # ← full PR curve, NOT filtered

# Step 2: Precision/Recall/F1/FP/FN — at F1-optimal operating threshold
final_res, final_mf1 = evaluate_at_threshold(
    final_preds, GT,
    conf_thresholds=[best_crack_conf, best_hotspot_conf]
)

# ── Final Report ──────────────────────────────────────────────────────
print()
print(f'  mAP@0.5        = {final_map50:.4f}   ← full PR curve (conf→0)')
print(f'  Crack  AP@0.5  = {final_aps[0]:.4f}')
print(f'  Hotspot AP@0.5 = {final_aps[1]:.4f}')
print(f'  Crack  Prec    = {final_res[0]["P"]:.4f}   ← at conf={best_crack_conf}')
print(f'  Hotspot Prec   = {final_res[1]["P"]:.4f}   ← at conf={best_hotspot_conf}')
print(f'  Crack  FP/FN   = {final_res[0]["FP"]}/{final_res[0]["FN"]}')
print(f'  Hotspot FP/FN  = {final_res[1]["FP"]}/{final_res[1]["FN"]}')

print()
print('╔══════════════════════════════════════════════════════════════════════╗')
print('║  FADNET FIXED MASTER METRICS (F1-Optimal Operating Point)          ║')
print('╠════════════════╦══════════╦═══════════╦══════════╦════╦════╦════╦═══╣')
print('║ Class          ║  AP@0.5  ║ Precision ║  Recall  ║ TP ║ FP ║ FN ║ GT║')
print('╠════════════════╬══════════╬═══════════╬══════════╬════╬════╬════╬═══╣')
for c, name in enumerate(CLASS_NAMES):
    r = final_res[c]
    ap = final_aps.get(c, 0)
    print(f'║ {name:<14} ║  {ap:.4f}  ║   {r["P"]:.4f}  ║  {r["R"]:.4f}  '
          f'║{r["TP"]:>4}║{r["FP"]:>4}║{r["FN"]:>4}║{r["GT"]:>3}║')
print('╠════════════════╩══════════╩═══════════╩══════════╩════╩════╩════╩═══╣')
print(f'║  Final mAP@0.5 = {final_map50:.4f}     mean F1 = {final_mf1:.4f}                      ║')
print('╚══════════════════════════════════════════════════════════════════════╝')
print(f'\n  Before fix: Crack FP=70, Hotspot FP=143')
print(f'  After fix:  Crack FP={final_res[0]["FP"]}, Hotspot FP={final_res[1]["FP"]}')
print(f'  FP reduction: Crack {70-final_res[0]["FP"]:+d}, Hotspot {143-final_res[1]["FP"]:+d}')


In [ ]:
# ==============================================================================
# CELL 20 — F1 Curve Plots
# ==============================================================================
# ══════════════════════════════════════════════════════════════════════
# F1 curve plots
# ══════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, curve, name, best_conf, col in [
    (axes[0], crack_f1_curve,   'Crack',   best_crack_conf,   'steelblue'),
    (axes[1], hotspot_f1_curve, 'Hotspot', best_hotspot_conf, 'tomato'),
]:
    ax.plot(CONF_SWEEP, curve, 'o-', color=col, lw=2)
    ax.axvline(best_conf, color='k', ls='--', lw=1.5,
               label=f'F1-optimal conf={best_conf:.2f}')
    ax.set_title(f'{name} — F1 vs confidence threshold\n(post-WBF operating point)')
    ax.set_xlabel('Confidence threshold')
    ax.set_ylabel('F1 score')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('F1-optimal threshold per class — fixes FP bleed from SAHI', fontsize=11)
plt.tight_layout()
plt.savefig('/kaggle/working/f1_optimal_curves.png', dpi=130)
plt.show()

print(f'\nFinal settings for inference / paper reporting:')
print(f'  crack_conf    = {best_crack_conf}')
print(f'  hotspot_conf  = {best_hotspot_conf}')
print(f'  mAP@0.5       = {final_map50:.4f}  (paper metric)')
print(f'  mean F1       = {final_mf1:.4f}   (demo metric)')

In [ ]:
import zipfile
import os
from datetime import datetime

# 1. Define the specific files we created during this session
files_to_archive = [
    # Heatmap from Lever 1
    '/kaggle/working/perclass_thresh_heatmap.png', 
    # The progress chart from Cell 15
    '/kaggle/working/fadnet_advanced_push.png', 
    # The F1-Optimal curves from Cell 20
    '/kaggle/working/f1_optimal_curves.png', 
    # The primary weights used for FADNet
    '/kaggle/input/datasets/vishokbadri/latestrun/fadnet_finetune_best.pt',
    # The fixed YAML config
    '/kaggle/working/Thermal-H&C-1/data.yaml'
]

# 2. Create a timestamped filename so you don't overwrite previous saves
timestamp = datetime.now().strftime('%Y%m%d_%H%M')
zip_name = f"FADNet_F1_Optimized_Backup_{timestamp}.zip"

print(f"📦 Starting archive: {zip_name}")

with zipfile.ZipFile(zip_name, 'w') as archive:
    for file_path in files_to_archive:
        if os.path.exists(file_path):
            # Save the file using just its name, not the full path
            archive.write(file_path, arcname=os.path.basename(file_path))
            print(f"  + Added: {os.path.basename(file_path)}")
        else:
            print(f"  ⚠️ Warning: {os.path.basename(file_path)} not found in path.")

print(f"\n✅ All set, Ash! You can find '{zip_name}' in the Kaggle 'Output' sidebar.")

In [ ]:
from IPython.display import FileLink
import os

# Identify the zip file in the working directory
files = [f for f in os.listdir('/kaggle/working/') if f.endswith('.zip')]

if files:
    # Generates a clickable link for the most recent zip
    display(FileLink(files[-1]))
else:
    print("No zip file found in /kaggle/working/")